# VAETIMEGAN WTI

In [1]:
import pandas as pd
import numpy as np
import itertools
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import importlib
import tensorflow as tf
from tensorflow.keras import backend as K

!git clone https://github.com/Meowth239/dse4101-project.git
!git -C /content/dse4101-project checkout LSTM_r
import sys
sys.path.append('/content/dse4101-project/lstm_src')

import my_lstm
importlib.reload(my_lstm)
import gc

from my_lstm import build_lstm_model, create_sequences, expanding_window_lstm_forecast

Cloning into 'dse4101-project'...
remote: Enumerating objects: 456, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 456 (delta 70), reused 55 (delta 37), pack-reused 352 (from 3)
Receiving objects: 100% (456/456), 91.07 MiB | 19.50 MiB/s, done.
Resolving deltas: 100% (210/210), done.
Branch 'LSTM_r' set up to track remote branch 'LSTM_r' from 'origin'.
Switched to a new branch 'LSTM_r'


In [2]:
#df = pd.read_csv('../data/final_data_with_VAETIMEGAN.csv')
df = pd.read_csv('/content/dse4101-project/data/final_data_with_VAETIMEGAN.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.head()

,Date,AUD_USD_ret,CAD_USD_ret,NZD_USD_ret,ZAR_USD_ret,CPI,TB3MS,M1,M2,wti_ret
0,1958-03-01,0.012299,-0.002406,-0.006743,-0.023616,-0.005484,-0.073454,0.013188,0.008006,-0.203766
1,1958-04-01,0.020833,-0.009353,0.060724,0.005213,-0.003673,-0.079281,0.011769,0.009704,0.018075
2,1958-05-01,0.006006,-0.008792,0.009556,0.067115,0.002756,0.014742,0.017015,0.010501,0.180503
3,1958-06-01,-0.054147,0.010266,-0.037406,0.144931,0.003663,0.009709,0.013104,0.007785,-0.135577
4,1958-07-01,-0.090823,-0.006553,-0.026149,0.010176,0.000914,-0.063144,0.014577,0.008258,-0.147348


In [3]:
feature_cols = [
    "wti_ret"
]

target_col = "wti_ret"

In [4]:
df = df.sort_values("Date").reset_index(drop=True)

# compute indices based on original evaluation and test split
train_size = df.index[df["Date"] < "2014-02-01"][-1] + 1
val_end = df.index[df["Date"] <= "2020-01-01"][-1] + 1

# split using iloc
train_data = df.iloc[:train_size]
val_data = df.iloc[train_size:val_end]
test_data = df.iloc[val_end:]

print(f"Train data length: {len(train_data)}")
print(f"Validation data length: {len(val_data)}")
print(f"Test data length: {len(test_data)}")

Train data length: 671
Validation data length: 72
Test data length: 72


In [5]:
param_grid_tiny = {
    "lookback": [2, 10],
    "dropout": [0.001, 0.1],
    "units": [50, 170],
    "epochs": [50, 100]
}

param_combinations = list(itertools.product(
    param_grid_tiny["lookback"],
    param_grid_tiny["dropout"],
    param_grid_tiny["units"],
    param_grid_tiny["epochs"]
))

results_grid = []
failed_combos = []

max_retries = 2

for i, (lb, dr, units, ep) in enumerate(param_combinations, 1):
    print(f"\n[{i}/{len(param_combinations)}] Testing params:")
    print(f"lookback={lb}, dropout={dr}, units={units}, epochs={ep}")

    success = False

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}/{max_retries}")

            # clear memory before each attempt
            K.clear_session()
            tf.keras.backend.clear_session()
            gc.collect()

            val_forecasts = expanding_window_lstm_forecast(
            df=df,
            feature_cols=feature_cols,
            target_col=target_col,
            initial_train_size=train_size,
            end_idx=val_end,
            date_col="Date",
            lookback=lb,
            units=units,
            dropout=dr,
            epochs=ep,
            batch_size=32,
            verbose=0,
            scale=True,
            seed=42
            )

            if len(val_forecasts) == 0:
                print("No forecasts generated.")
                raise ValueError("No forecasts generated")

            mse = mean_squared_error(
                val_forecasts["actual"],
                val_forecasts["predicted"]
            )

            print(f"Validation MSE: {mse:.6f}")

            results_grid.append({
                "lookback": lb,
                "dropout": dr,
                "units": units,
                "epochs": ep,
                "mse": mse,
                "attempt_used": attempt
            })

            success = True
            break

        except Exception as e:
            print(f"Error on attempt {attempt}: {e}")

            # clear memory after failure
            K.clear_session()
            tf.keras.backend.clear_session()
            gc.collect()

    if not success:
        print("Failed after all retries. Skipping this combination.")
        failed_combos.append({
            "lookback": lb,
            "dropout": dr,
            "units": units,
            "epochs": ep
        })

# --------------------------------------------------
# results
# --------------------------------------------------

results_grid_df = pd.DataFrame(results_grid).sort_values("mse").reset_index(drop=True)
failed_combos_df = pd.DataFrame(failed_combos)


[1/16] Testing params:
lookback=2, dropout=0.001, units=50, epochs=50
Attempt 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a l

Validation MSE: 0.087031

[2/16] Testing params:
lookback=2, dropout=0.001, units=50, epochs=100
Attempt 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a l

Validation MSE: 0.092643

[3/16] Testing params:
lookback=2, dropout=0.001, units=170, epochs=50
Attempt 1/2


...

Validation MSE: 0.084922

[9/16] Testing params:
lookback=10, dropout=0.001, units=50, epochs=50
Attempt 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a l

Validation MSE: 0.181897

[10/16] Testing params:
lookback=10, dropout=0.001, units=50, epochs=100
Attempt 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
Exception ignored in: <function WeakKeyDictionary.__init__.<locals>.remove at 0x7a6a00e1ac00>
Traceback (most recent call last):
  File "/usr/lib/python3.12/weakref.py", line 369, in remove
    def remove(k, selfref=ref(self)):

KeyboardInterrupt: 
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
Exception ignored in: <function _xla_gc_callback at 0x7a69fb64aa20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", l

KeyboardInterrupt: 

In [6]:
results_grid

[{'lookback': 2,
  'dropout': 0.001,
  'units': 50,
  'epochs': 50,
  'mse': 0.08703139957276582,
  'attempt_used': 1},
 {'lookback': 2,
  'dropout': 0.001,
  'units': 50,
  'epochs': 100,
  'mse': 0.0926430972724055,
  'attempt_used': 1},
 {'lookback': 2,
  'dropout': 0.001,
  'units': 170,
  'epochs': 50,
  'mse': 0.07934547800064226,
  'attempt_used': 1},
 {'lookback': 2,
  'dropout': 0.001,
  'units': 170,
  'epochs': 100,
  'mse': 0.07970318571584396,
  'attempt_used': 1},
 {'lookback': 2,
  'dropout': 0.1,
  'units': 50,
  'epochs': 50,
  'mse': 0.08908143495317353,
  'attempt_used': 1},
 {'lookback': 2,
  'dropout': 0.1,
  'units': 50,
  'epochs': 100,
  'mse': 0.08908143495317353,
  'attempt_used': 1},
 {'lookback': 2,
  'dropout': 0.1,
  'units': 170,
  'epochs': 50,
  'mse': 0.0849222911937075,
  'attempt_used': 1},
 {'lookback': 2,
  'dropout': 0.1,
  'units': 170,
  'epochs': 100,
  'mse': 0.0849222911937075,
  'attempt_used': 1},
 {'lookback': 10,
  'dropout': 0.001,
  'un

In [ ]:
print("\nTop results:")
print(results_grid_df)

print("\nFailed combinations:")
print(failed_combos_df)

In [ ]:
best_params = results_grid_df.iloc[0]
best_lb = int(best_params["lookback"])
best_dr = float(best_params["dropout"])
best_units = int(best_params["units"])
best_ep = int(best_params["epochs"])

best_params

In [ ]:
test_results = expanding_window_lstm_forecast(
    df=df,
    feature_cols=feature_cols,
    target_col=target_col,
    initial_train_size=val_end,
    end_idx=len(df),
    date_col="Date",
    lookback=int(best_params["lookback"]),
    units=int(best_params["units"]),
    dropout=float(best_params["dropout"]),
    epochs=int(best_params["epochs"]),
    batch_size=32,
    verbose=0,
    scale=True,
    seed=42
)

test_mse = mean_squared_error(
    test_results["actual"],
    test_results["predicted"]
)

test_mape = mean_absolute_percentage_error(
    test_results["actual"],
    test_results["predicted"]
)

test_r2 = r2_score(
    test_results["actual"],
    test_results["predicted"]
)

print("Test MSE:", test_mse)
print("Test MAPE:", test_mape)
print("Test R²:", test_r2)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(test_results['Date'], test_results["actual"], label="Actual")
plt.plot(test_results['Date'], test_results["predicted"], label="Predicted")
plt.legend()
plt.title("Out-of-sample Forecast for LSTM-TIMEGAN (With only wti)")
plt.show()

In [ ]:
# save results to csv
#test_results.to_csv('results/lstm_test_vaetimegan_results.csv', index = False)



from google.colab import drive
drive.mount('/content/drive')

import os

base_path = '/content/drive/MyDrive/dse4101'
os.makedirs(base_path, exist_ok=True)



results_folder = os.path.join(base_path, 'results')
os.makedirs(results_folder, exist_ok=True)

# define file path
test_results_path = os.path.join(results_folder, 'lstm_test_vaetimegan_results.csv')

# save
test_results.to_csv(test_results_path, index=False)

print("Saved to:", test_results_path)